# Amazon Product Recommendation System - Interactive Exploration

This notebook provides an interactive walkthrough of the recommendation system pipeline.

## Table of Contents

1. Setup and Configuration
2. Data Ingestion from HDFS
3. Exploratory Data Analysis
4. Feature Engineering
5. Model Training
6. Evaluation
7. Generate Recommendations

## 1. Setup and Configuration

In [ ]:
# Import required libraries
import sys
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, mean, stddev
import matplotlib.pyplot as plt
import seaborn as sns

# Add project to path
sys.path.insert(0, '/home/sameer/BDA_PROJECT_NEW/amazon-recommendation-system')

# Initialize Spark session
spark = SparkSession.builder \
    .appName("Amazon Recommendation System") \
    .config("spark.executor.memory", "4G") \
    .config("spark.driver.memory", "2G") \
    .getOrCreate()

print(f"✓ Spark session initialized: {spark.version}")

## 2. Data Ingestion from HDFS

In [ ]:
# Load data directly from HDFS
df = spark.read.option("header", "true") \
               .option("inferSchema", "true") \
               .csv("hdfs://master:9000/dataset/all_csv_files.csv")

print(f"Dataset loaded: {df.count():,} rows")
print(f"Number of partitions: {df.rdd.getNumPartitions()}")
df.printSchema()

In [ ]:
# Display sample data
df.show(10)

## 3. Exploratory Data Analysis

In [ ]:
# Basic statistics
from pyspark.sql.functions import countDistinct

total_users = df.select(countDistinct('user_id')).collect()[0][0]
total_products = df.select(countDistinct('product_id')).collect()[0][0]
total_ratings = df.count()

print(f"Total Users: {total_users:,}")
print(f"Total Products: {total_products:,}")
print(f"Total Ratings: {total_ratings:,}")
print(f"Avg Ratings per User: {total_ratings / max(total_users, 1):.2f}")
print(f"Avg Ratings per Product: {total_ratings / max(total_products, 1):.2f}")

In [ ]:
# Rating distribution
rating_stats = df.select(
    mean('rating').alias('mean'),
    stddev('rating').alias('stddev'),
    min(col('rating')).alias('min'),
    max(col('rating')).alias('max')
).collect()[0]

print(f"Mean Rating: {rating_stats['mean']:.4f}")
print(f"Std Deviation: {rating_stats['stddev']:.4f}")
print(f"Min Rating: {rating_stats['min']}")
print(f"Max Rating: {rating_stats['max']}")

In [ ]:
# Top 10 most rated products
top_products = df.groupBy('product_id') \
    .agg(count('rating').alias('review_count'), 
         mean('rating').alias('avg_rating')) \
    .orderBy(col('review_count').desc()) \
    .limit(10)

top_products.show()

## 4. Data Preprocessing

In [ ]:
# Clean data
from pyspark.sql.functions import dropDuplicates

# Select required columns
df_clean = df.select('user_id', 'product_id', 'rating')

# Drop nulls
df_clean = df_clean.dropna()

# Remove duplicates
df_clean = df_clean.dropDuplicates()

# Filter valid ratings (0.5 to 5.0)
df_clean = df_clean.filter((col('rating') >= 0.5) & (col('rating') <= 5.0))

print(f"Cleaned dataset: {df_clean.count():,} ratings")
df_clean.cache()

## 5. Feature Engineering

In [ ]:
# User activity features
user_features = df_clean.groupBy('user_id') \
    .agg(
        count('rating').alias('user_activity_score'),
        mean('rating').alias('user_avg_rating')
    )

# Product popularity features
product_features = df_clean.groupBy('product_id') \
    .agg(
        count('rating').alias('product_popularity'),
        mean('rating').alias('product_avg_rating')
    )

print(f"Users: {user_features.count():,}")
print(f"Products: {product_features.count():,}")

## 6. Model Training

In [ ]:
from pyspark.ml.recommendation import ALS

# Split data
train_df, test_df = df_clean.randomSplit([0.8, 0.2], seed=42)

print(f"Training samples: {train_df.count():,}")
print(f"Test samples: {test_df.count():,}")

In [ ]:
# Configure and train ALS model
als = ALS(
    userCol="user_id",
    itemCol="product_id",
    ratingCol="rating",
    coldStartStrategy="drop",
    nonnegative=True,
    rank=10,
    maxIter=15,
    regParam=0.1
)

print("Training ALS model...")
model = als.fit(train_df)
print("✓ Model trained successfully!")

## 7. Model Evaluation

In [ ]:
from pyspark.ml.evaluation import RegressionEvaluator

# Generate predictions
predictions = model.transform(test_df)
valid_predictions = predictions.filter(predictions.prediction.isNotNull())

# Compute RMSE
evaluator = RegressionEvaluator(
    metricName="rmse",
    labelCol="rating",
    predictionCol="prediction"
)

rmse = evaluator.evaluate(valid_predictions)
print(f"RMSE: {rmse:.4f}")

# Compute MAE
mae = evaluator.setMetricName("mae").evaluate(valid_predictions)
print(f"MAE: {mae:.4f}")

## 8. Generate Recommendations

In [ ]:
# Get sample user recommendations
sample_user = df_clean.select('user_id').distinct().first()['user_id']
print(f"Generating recommendations for User {sample_user}")

# Get items user hasn't rated
user_rated = df_clean.filter(df_clean.user_id == sample_user).select('product_id')
all_items = df_clean.select('product_id').distinct()
candidate_items = all_items.subtract(user_rated)

# Create user-item pairs for prediction
user_candidates = df_clean.select('user_id').distinct() \
    .filter(col('user_id') == sample_user) \
    .crossJoin(candidate_items)

# Generate predictions
recs = model.transform(user_candidates)
top_recs = recs.orderBy(col('prediction').desc()).limit(10)

print(f"\nTop 10 Recommended Products for User {sample_user}:")
top_recs.select('user_id', 'product_id', 'prediction').show()

## 9. Visualization

In [ ]:
# Convert to pandas for plotting
pdf = df_clean.sample(0.1).toPandas()

# Plot rating distribution
plt.figure(figsize=(12, 6))
sns.histplot(data=pdf, x='rating', kde=True, bins=20, color='skyblue')
plt.title('Rating Distribution')
plt.xlabel('Rating')
plt.ylabel('Frequency')
plt.tight_layout()
plt.savefig('/tmp/rating_distribution.png', dpi=300)
plt.show()
print("✓ Plot saved to /tmp/rating_distribution.png")

## 10. Cleanup

In [ ]:
# Stop Spark session
spark.stop()
print("✓ Spark session stopped")